# 05 — Rapport mensuel et aide à la décision (Projet 4)

Ce notebook exécute le **pipeline mensuel** (données → features → signaux → Momentum → Volatility Targeting → métriques → alertes) et produit deux sorties : un **rapport Markdown** lisible et un **snapshot JSON** structuré (futur contrat de données pour une API / application / widget).

> Aucune logique n'est recodée ici : tout vient de `src/`. Aucun ordre n'est exécuté ; la décision reste humaine (paper trading).

## 1. Initialisation

In [1]:
import sys
from pathlib import Path


def find_project_root(markers=("src", "config")) -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if all((candidate / m).is_dir() for m in markers):
            return candidate
    raise RuntimeError("Racine du projet introuvable.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Racine :", PROJECT_ROOT)

Racine : C:\Users\degar\OneDrive - De Vinci Higher Education (DVHE)\Perso\Projet Investissement\personal-quant-lab


## 2. Configuration

In [2]:
import json
import pandas as pd

from src.monthly_pipeline import load_config, run_monthly_pipeline
from src.reporting import generate_markdown_report, save_report_snapshot, load_previous_snapshot

config = load_config(PROJECT_ROOT / "config" / "strategy_params.yaml")
print("Benchmark :", config.benchmark_ticker)
print("Momentum :", config.momentum)
print("Volatility targeting :", config.volatility_targeting)

Benchmark : CW8.PA
Momentum : MomentumConfig(lookback_months=12, skip_months=1, moving_average_window=10, top_n=1, require_positive_momentum=False)
Volatility targeting : VolTargetConfig(enabled=True, volatility_window=3, target_volatility=0.1, min_exposure=0.0, max_exposure=1.0)


## 3. Chargement des données

Données locales du Projet 2/3 (`momentum_universe.csv`). Le téléchargement n'est pas déclenché depuis le notebook (reproductibilité).

In [3]:
from src.data_loader import load_prices, download_prices, validate_prices, save_prices, load_tickers_from_yaml

DATA = PROJECT_ROOT / "data" / "processed" / "momentum_universe.csv"
if DATA.exists():
    prices = load_prices(DATA)
    print("Chargé :", DATA.relative_to(PROJECT_ROOT), "| forme", prices.shape)
else:
    etfs = [t for t in load_tickers_from_yaml(PROJECT_ROOT / "config" / "assets.yaml") if t.endswith(".PA")]
    prices = validate_prices(download_prices(etfs, start="2015-01-01"))
    save_prices(prices, DATA)
    print("Téléchargé et sauvegardé.")
prices.tail(3)

Chargé : data\processed\momentum_universe.csv | forme (2947, 3)


,CW8.PA,EWLD.PA,ESE.PA
Date,,,
2026-07-06,687.739990,40.744999,33.4165
2026-07-07,682.530029,40.435001,33.1716
2026-07-08,677.539978,40.160000,33.0215


## 4. Exécution du pipeline

On fixe l'horodatage de génération juste après la dernière observation pour un résultat reproductible (évite une alerte de données « anciennes » liée à la date d'exécution).

In [4]:
generation_time = prices.index.max() + pd.Timedelta(days=1)
result = run_monthly_pipeline(prices, config, generation_time=generation_time, data_refreshed=False)
report = result.report
report.metadata

{'schema_version': '1.0',
 'report_month': '2026-07',
 'report_date': '2026-07-08',
 'signal_date': '2026-06-30',
 'last_observation': '2026-07-08',
 'generated_at': '2026-07-09T00:00:00',
 'history_start': '2015-01-31',
 'history_end': '2026-06-30',
 'reference_currency': 'EUR',
 'data_mode': 'local',
 'pipeline_status': 'OK_WITH_WARNINGS'}

## 5. Inspection du résultat structuré

In [5]:
print("Statut :", report.metadata["pipeline_status"])
print("Date du signal :", report.metadata["signal_date"], "(dernier mois complet)")
print("Univers :", report.universe["assets"], "| mois :", report.universe["n_months"])

Statut : OK_WITH_WARNINGS
Date du signal : 2026-06-30 (dernier mois complet)
Univers : ['CW8.PA', 'ESE.PA'] | mois : 138


## 6. Qualité des données

In [6]:
dq = report.data_quality
print("Demandés :", dq["n_requested"], "| retenus :", dq["n_kept"])
print("Exclus :", dq["excluded_assets"])
print("Ancienneté (jours) :", dq["data_age_days"], "| mois partiel :", dq["partial_month"])
pd.Series(dq["last_date_per_asset"])

Demandés : 3 | retenus : 2
Exclus : {'EWLD.PA': 'historique insuffisant (29 mois < 34)'}
Ancienneté (jours) : 1 | mois partiel : 2026-07


CW8.PA    2026-07-08
ESE.PA    2026-07-08
dtype: object

## 7. Signaux récents

In [7]:
mom = report.momentum_strategy
print("Momentum sélectionné :", mom.selected_assets)
print("Momentum (valeurs) :", {k: round(v, 3) if v is not None else None for k, v in mom.momentum.items()})
print("Filtre de tendance :", mom.trend_filter)
print("Volatilité réalisée :", round(mom.realized_volatility, 4) if mom.realized_volatility else None)

Momentum sélectionné : ['ESE.PA']
Momentum (valeurs) : {'CW8.PA': 0.229, 'ESE.PA': 0.242}
Filtre de tendance : {'CW8.PA': True, 'ESE.PA': True}
Volatilité réalisée : 0.1195


## 8. Allocations (théoriques, à valider)

In [8]:
def weights_frame(strat):
    return pd.Series(strat.target_weights, name=strat.name)

alloc = pd.concat([
    weights_frame(report.momentum_strategy),
    weights_frame(report.volatility_targeted_strategy),
], axis=1).fillna(0.0)
print("Allocation cible (poids) :")
print((alloc * 100).round(1).astype(str) + " %")

Allocation cible (poids) :
       Momentum défensif 12-1 + MA10 Momentum + Volatility Targeting
CW8.PA                         0.0 %                           0.0 %
ESE.PA                       100.0 %                          83.7 %
CASH                           0.0 %                          16.3 %


## 9. Comparaison Momentum / VT

In [9]:
comp = report.allocation_comparison
print("Exposition investie — Momentum :", f"{comp['momentum_invested_exposure']:.0%}",
      "| VT :", f"{comp['volatility_targeted_invested_exposure']:.0%}")
print("Écart d'exposition :", f"{comp['exposure_difference']:.1%}")
vt = report.volatility_targeted_strategy
print(f"VT — CAGR {vt.cagr:.2%} | Vol {vt.annualized_volatility:.2%} | Sharpe {vt.sharpe:.2f} | MaxDD {vt.max_drawdown:.2%}")

Exposition investie — Momentum : 100% | VT : 84%
Écart d'exposition : 16.3%
VT — CAGR 7.20% | Vol 9.88% | Sharpe 0.75 | MaxDD -14.93%


## 10. Alertes

In [10]:
for a in report.alerts:
    print(f"[{a.level}] {a.code} — {a.message}")
if not report.alerts:
    print("Aucune alerte.")

[INFO] DATA_NOT_REFRESHED — Rapport basé sur les données locales (pas de téléchargement).
[WARNING] ASSET_EXCLUDED — Actifs écartés faute d'historique : EWLD.PA.
[INFO] PARTIAL_MONTH — Le mois 2026-07 est incomplet et exclu du signal officiel.


## 11. Génération du rapport Markdown

In [11]:
OUT = PROJECT_ROOT / "reports" / "monthly"
month = report.metadata["report_month"]
previous = load_previous_snapshot(OUT, month)
md_path = generate_markdown_report(result, OUT / f"{month}-report.md")
print("Rapport Markdown :", md_path.relative_to(PROJECT_ROOT))

Rapport Markdown : reports\monthly\2026-07-report.md


## 12. Génération du snapshot JSON

In [12]:
json_path = save_report_snapshot(result, OUT / f"{month}-report.json")
snapshot = json.loads(json_path.read_text(encoding="utf-8"))
print("Snapshot JSON :", json_path.relative_to(PROJECT_ROOT))
print("Clés :", list(snapshot.keys()))
print("Décision manuelle :", snapshot["manual_decision"]["status"])

Snapshot JSON : reports\monthly\2026-07-report.json
Clés : ['schema_version', 'metadata', 'data_quality', 'universe', 'benchmark', 'momentum_strategy', 'volatility_targeted_strategy', 'allocation_comparison', 'risk', 'alerts', 'manual_decision', 'previous_comparison']
Décision manuelle : PENDING


## 13. Extrait du rapport

In [13]:
extract = md_path.read_text(encoding="utf-8").splitlines()
print("\n".join(extract[:30]))

# Rapport mensuel — 2026-07

_Ce rapport est produit à des fins éducatives et de paper trading._
_Aucun ordre n'est exécuté ; la décision finale reste humaine._

## Statut général

- Statut du pipeline : **OK_WITH_WARNINGS**
- Date d'arrêté (report_date) : 2026-07-08
- Date effective du signal : **2026-06-30** (dernier mois civil complet)
- Dernière observation disponible : 2026-07-08
- Généré le : 2026-07-09T00:00:00
- Devise de référence : EUR | Mode : local
- Schéma : v1.0

## Qualité des données

- Actifs demandés : 3 | retenus : 2
- Actifs exclus : EWLD.PA (historique insuffisant (29 mois < 34))
- Ancienneté des données : 1 jours
- Mois partiel exclu du signal : 2026-07
- Avertissements : Actifs écartés faute d'historique : EWLD.PA.

## Univers retenu

- Benchmark : CW8.PA
- Actifs : CW8.PA, ESE.PA
- Période : 2015-01-31 → 2026-06-30 (138 mois)

## Benchmark


## 14. Validation finale

In [14]:
# Poids valides et somme = 1
for strat in (report.momentum_strategy, report.volatility_targeted_strategy):
    w = strat.target_weights
    assert all(-1e-9 <= v <= 1 + 1e-9 for v in w.values()), "Poids hors [0, 1]."
    assert abs(sum(w.values()) - 1.0) < 1e-6, "Somme des poids != 1."

# Statut non-échec, décision PENDING, JSON sans NaN, aucun ordre
assert report.metadata["pipeline_status"] in ("OK", "OK_WITH_WARNINGS")
assert snapshot["manual_decision"]["status"] == "PENDING"
raw = json_path.read_text(encoding="utf-8")
assert "NaN" not in raw and "Infinity" not in raw
report_text = md_path.read_text(encoding="utf-8").lower()
assert not any(w in report_text for w in ("achetez", "vendez", "ordre à exécuter"))
# Aucune date future utilisée
assert pd.Timestamp(report.metadata["signal_date"]) <= prices.index.max()
print("Tous les contrôles passent.")

Tous les contrôles passent.


In [15]:
print("Projet 4 — Rapport mensuel validé techniquement.")
print("Le rapport reste un outil d'aide à la décision en paper trading.")

Projet 4 — Rapport mensuel validé techniquement.
Le rapport reste un outil d'aide à la décision en paper trading.
